# Quickstart

Train a synthetic data model, generate rows, and reload the trained model later.

Every Katabatic model shares one interface, so this notebook works for any of them: change `MODEL` and install that model's extra, e.g. `pip install "katabatic[ctgan]"`. NaiveBayes needs only the core install.

In [1]:
from katabatic.artifacts import LocalArtifactStore
from katabatic.models import ModelRegistry, get_model, list_supported_models
from katabatic.pipeline import TrainTestSplitPipeline

list_supported_models()

['ganblr',
 'great',
 'realtabformer',
 'tabsyn',
 'pategan',
 'mst',
 'ctgan',
 'kde',
 'arf',
 'privtree',
 'synthpop',
 'naivebayes',
 'histogram',
 'fairtabdiffusion',
 'smote']

## Data

Any CSV works, with the target in the last column. Here we use the car evaluation dataset that ships with Katabatic.

In [2]:
from importlib.resources import files
from pathlib import Path

import pandas as pd

car = pd.read_csv(files("katabatic.datasets") / "car.csv")
car.columns = ["buying", "maint", "doors", "persons", "lug_boot", "safety", "class"]
Path("artifacts").mkdir(exist_ok=True)
car.to_csv("artifacts/car.csv", index=False)
car.head()

,buying,maint,doors,persons,lug_boot,safety,class
0,vhigh,vhigh,2,2,small,low,unacc
1,vhigh,vhigh,2,2,small,med,unacc
2,vhigh,vhigh,2,2,small,high,unacc
3,vhigh,vhigh,2,2,med,low,unacc
4,vhigh,vhigh,2,2,med,med,unacc


## Train

`TrainTestSplitPipeline` splits the data, trains the model, writes synthetic data, and scores it with TSTR (train a classifier on synthetic data, test on real data). The artifact store keeps each step versioned on disk, so any run can be reloaded or re-evaluated.

In [3]:
MODEL = "naivebayes"

model = get_model(MODEL)
store = LocalArtifactStore("artifacts")
results = TrainTestSplitPipeline(model=model).run(
    input_csv="artifacts/car.csv",
    dataset_name="car",
    artifact_store=store,
    model_name=MODEL,
    target_column="class",
)
model_ref = results["model_ref"]
model_ref.root_relpath

Loaded data with shape: (1728, 7)
Train label distribution:
 class
unacc    0.700434
acc      0.222142
good     0.039797
vgood    0.037627
Name: proportion, dtype: float64
Test label distribution:
 class
unacc    0.699422
acc      0.222543
good     0.040462
vgood    0.037572
Name: proportion, dtype: float64
Saved dataset artifact under datasets/car/split-20260925-114446
[NaiveBayes] Synthetic data saved to: artifacts/models/naivebayes_car_train-20260925-114446/synthetic


/home/adity/.cache/pypoetry/virtualenvs/katabatic--vdtbdaF-py3.11/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(



Results saved to: artifacts/evaluations/naivebayes_car_train-20260925-114446/tstr_report.csv

TSTR Evaluation Results:

LR:
Accuracy: 0.8584
F1 Score: 0.8513

MLP:
Accuracy: 0.8064
F1 Score: 0.7932

RF:
Accuracy: 0.7919
F1 Score: 0.7796

XGBoost:
Accuracy: 0.7861
F1 Score: 0.7783


'models/naivebayes_car_train-20260925-114446'

In [6]:
tstr = store.load_json(results["evaluation_refs"][0].metrics_relpath)
pd.DataFrame(tstr).round(4)

,LR,MLP,RF,XGBoost
Accuracy,0.8584,0.8064,0.7919,0.7861
F1 Score,0.8513,0.7932,0.7796,0.7783


## Generate

In [7]:
model.sample(5)

,buying,maint,doors,persons,lug_boot,safety,class
0,low,high,4,2,small,low,unacc
1,high,low,4,2,big,low,unacc
2,vhigh,high,4,4,med,med,unacc
3,vhigh,low,2,2,big,low,unacc
4,vhigh,vhigh,3,4,big,med,acc


## Reload later

`model_ref` identifies the run. `load_from_ref()` rebuilds the fitted model from the store, in this session or a later one.

In [8]:
reloaded = ModelRegistry.load_model(MODEL).load_from_ref(store, model_ref)
reloaded.sample(5)

,buying,maint,doors,persons,lug_boot,safety,class
0,vhigh,low,5more,4,big,med,unacc
1,vhigh,vhigh,2,2,med,med,unacc
2,med,vhigh,3,2,med,high,unacc
3,vhigh,med,4,2,small,med,unacc
4,med,high,3,4,big,med,acc


## Next

- [evaluation.ipynb](evaluation.ipynb): score a model on fidelity, utility, diversity, privacy, consistency and stability.
- [remote_artifact_store.ipynb](remote_artifact_store.ipynb): share artifacts between machines through S3, GCS or Azure.